# 01 EDA — Predicting Smartphone Addiction

Load validated frames via the project package. No model training.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from smartphone_addiction.data.download import fingerprint_files
from smartphone_addiction.data.load import load_competition_frames
from smartphone_addiction.data.schema import FEATURE_COLUMNS, NUMERIC_COLUMNS
from smartphone_addiction.paths import project_root

ROOT = project_root()
RAW = ROOT / "data" / "raw"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

frames = load_competition_frames(RAW)
digests = fingerprint_files(RAW)
train, test = frames.train, frames.test
print({"train": train.shape, "test": test.shape})
print("target_rate", float(train["addicted_label"].mean()))
print(digests)

In [ ]:
# Target and screen-time distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train["addicted_label"].value_counts(normalize=True).plot(
    kind="bar", ax=axes[0], title="Target share"
)
train["daily_screen_time_hours"].hist(bins=40, ax=axes[1])
axes[1].set_title("daily_screen_time_hours")
fig.tight_layout()
fig.savefig(FIG / "eda_target_screen.png", dpi=120)
plt.close(fig)

In [ ]:
# Missingness and train/test numeric means
miss = pd.DataFrame(
    {
        "train": train[FEATURE_COLUMNS].isna().sum(),
        "test": test[FEATURE_COLUMNS].isna().sum(),
    }
).sort_values("train", ascending=False)
display(miss.head(12))

cmp = pd.DataFrame(
    {
        "train_mean": train[NUMERIC_COLUMNS].mean(),
        "test_mean": test[NUMERIC_COLUMNS].mean(),
    }
)
cmp["abs_diff"] = (cmp["train_mean"] - cmp["test_mean"]).abs()
cmp = cmp.sort_values("abs_diff", ascending=False)
cmp.to_csv(FIG / "eda_train_test_numeric_means.csv")
display(cmp.head(12))

# Prefer the script for the full official report:
# python scripts/write_data_validation_report.py